# Analisis Final: Explicando el Churn al Negocio

## Caso Practico - Empresa de Telecomunicaciones
## Practicas Aplicadas 2026

---

## Objetivo de este notebook

Hasta ahora hemos construido y evaluado varios modelos de prediccion de churn. En este notebook damos un paso mas: **convertir los resultados tecnicos en insights accionables para el negocio**.

El objetivo no es solo predecir quien se va, sino responder las preguntas que haria un director comercial:

- *Como es el cliente que abandona? Que perfil tiene?*
- *En que zonas se concentra el problema?*
- *Puedo meter los datos de un cliente concreto y saber su riesgo?*
- *Funciona mejor el modelo si lo entreno por tipo de plan?*

Este notebook contiene cuatro secciones:
1. **Perfil del churner tipico** — quien es el cliente que se va?
2. **Mapa de riesgo por zona** — donde se concentra el churn?
3. **Simulador de cliente** — cual es el riesgo de un cliente concreto?
4. **Modelo por segmento** — mejora el AUC si entrenamos un modelo por tipo de plan?


## 1. Librerias y carga de datos


In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.load import load_all
from src.clean import clean_all

RANDOM_STATE = 42
PAL = {'No Churn': '#4C9BE8', 'Churn': '#E85C4C'}
print('Librerias cargadas')


Librerias cargadas


In [ ]:
data  = load_all()
clean = clean_all(data, save=False)

clientes  = clean['clientes']
churn     = clean['churn']
factura   = clean['facturacion']
soporte   = clean['soporte']
calidad   = clean['calidad']

# Target: ever_churn por cliente
churn_agg = churn.groupby('cliente_id').agg(
    ever_churn         = ('churn', 'max'),
    n_meses_observados = ('churn', 'count'),
).reset_index()

# Facturacion agregada por cliente
factura['mes'] = factura['fecha'].dt.to_period('M').dt.to_timestamp()
fac_agg = factura.groupby('cliente_id').agg(
    importe_medio      = ('importe_total',     'mean'),
    pct_meses_impago   = ('impago_flag',       'mean'),
    dias_retraso_medio = ('dias_retraso_pago', 'mean'),
    stress_medio       = ('stress_calidad_lag','mean'),
    n_meses_facturados = ('importe_total',     'count'),
).reset_index()

# Soporte agregado por cliente
sop_agg = soporte.groupby('cliente_id').agg(
    n_interacciones    = ('interaccion_id',    'count'),
    satisfaccion_media = ('satisfaccion_post', 'mean'),
).reset_index()

# Tabla analitica completa
df = clientes.merge(churn_agg, on='cliente_id', how='inner')
df = df.merge(fac_agg, on='cliente_id', how='left')
df = df.merge(sop_agg, on='cliente_id', how='left')
df['churn_label'] = df['ever_churn'].map({0: 'No Churn', 1: 'Churn'})

print(f'Tabla analitica: {df.shape[0]:,} clientes')
print(f'Tasa de churn: {df["ever_churn"].mean()*100:.1f}%')


Tabla analitica: 10,150 clientes
Tasa de churn: 20.3%


---
## 2. Perfil del churner tipico

Antes de hablar de modelos y AUCs, la pregunta mas importante que puede hacer un director comercial es: **como es el cliente que se va?**

Esta seccion describe el perfil del churner de forma clara y accionable.


In [ ]:
# Tabla comparativa: churner vs no churner
vars_perfil = [
    'edad', 'antiguedad_meses', 'num_lineas', 'ingreso_estimado',
    'importe_medio', 'pct_meses_impago', 'dias_retraso_medio',
    'stress_medio', 'n_interacciones', 'satisfaccion_media'
]
vars_disp = [v for v in vars_perfil if v in df.columns]

perfil = df.groupby('ever_churn')[vars_disp].mean().T
perfil.columns = ['No Churn', 'Churn']
perfil['Diferencia (%)'] = (
    (perfil['Churn'] - perfil['No Churn']) / perfil['No Churn'].abs() * 100
).round(1)
perfil['Senal'] = perfil['Diferencia (%)'].apply(
    lambda x: 'Mayor en churners' if x > 5
    else ('Menor en churners' if x < -5 else 'Similar')
)

print('=== PERFIL MEDIO: Churner vs No Churner ===')
display(perfil.round(2))


In [ ]:
# Tasa de churn por variables categoricas clave
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

vars_cat = [
    ('tipo_plan', 'Tipo de plan',  ['Prepago', 'Contrato', 'Premium']),
    ('tipo_zona', 'Tipo de zona',  ['rural', 'suburbana', 'urbana_premium']),
    ('region',    'Region',        None),
]

for ax, (var, titulo, orden) in zip(axes, vars_cat):
    tasas = (
        df.groupby(var)['ever_churn']
        .agg(['mean', 'count'])
        .reset_index()
        .rename(columns={'mean': 'tasa', 'count': 'n'})
    )
    if orden:
        tasas = tasas.set_index(var).reindex(orden).reset_index()
    else:
        tasas = tasas.sort_values('tasa', ascending=False)

    bars = ax.bar(tasas[var].astype(str), tasas['tasa'] * 100,
                  color=sns.color_palette('RdYlGn_r', len(tasas)))
    ax.set_title(f'Tasa de churn por {titulo}', fontweight='bold')
    ax.set_ylabel('% clientes con churn')
    ax.tick_params(axis='x', rotation=15)
    for bar, (_, row) in zip(bars, tasas.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{row['tasa']*100:.1f}%\n(n={row['n']:,})",
            ha='center', va='bottom', fontsize=8
        )

plt.suptitle('Tasa de churn por segmento', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Descripcion automatica del perfil del churner
churners    = df[df['ever_churn'] == 1]
no_churners = df[df['ever_churn'] == 0]

plan_top   = churners['tipo_plan'].value_counts().index[0]
zona_top   = churners['tipo_zona'].value_counts().index[0]
region_top = churners['region'].value_counts().index[0]
edad_med   = churners['edad'].median()
antig_med  = churners['antiguedad_meses'].median()
impago_pct = churners['pct_meses_impago'].mean() * 100
stress_med = churners['stress_medio'].mean()

print('=' * 60)
print('  PERFIL DEL CLIENTE EN RIESGO DE CHURN')
print('=' * 60)
print(f'  Plan mas frecuente:     {plan_top}')
print(f'  Zona mas frecuente:     {zona_top}')
print(f'  Region mas frecuente:   {region_top}')
print(f'  Edad mediana:           {edad_med:.0f} anos')
print(f'  Antiguedad mediana:     {antig_med:.0f} meses')
print(f'  % meses con impago:     {impago_pct:.1f}%')
print(f'  Estres de red medio:    {stress_med:.3f}')
print('=' * 60)
print()
print(f'En resumen: el cliente que mas abandona es un cliente')
print(f'de plan {plan_top}, en zona {zona_top}, con menos de')
print(f'{antig_med:.0f} meses de antiguedad, que ha tenido impagos')
print(f'en el {impago_pct:.0f}% de sus meses y que vive en una zona')
print(f'con alto estres de red ({stress_med:.2f} sobre 1).')


  PERFIL DEL CLIENTE EN RIESGO DE CHURN
  Plan mas frecuente:     Premium
  Zona mas frecuente:     suburbana
  Region mas frecuente:   Oeste
  Edad mediana:           40 anos
  Antiguedad mediana:     24 meses
  % meses con impago:     14.8%
  Estres de red medio:    0.458

En resumen: el cliente que mas abandona es un cliente
de plan Premium, en zona suburbana, con menos de
24 meses de antiguedad, que ha tenido impagos
en el 15% de sus meses y que vive en una zona
con alto estres de red (0.46 sobre 1).


---
## 3. Mapa de riesgo por zona

Una empresa de telecomunicaciones opera en zonas geograficas. Saber **donde** se concentra el churn es tan importante como saber **quien** churna.

Esta seccion muestra el ranking de zonas por tasa de churn y la correlacion con el estres de red.


In [ ]:
# Metricas por zona
zona_stats = df.groupby('zona_id').agg(
    n_clientes       = ('cliente_id',       'count'),
    tasa_churn       = ('ever_churn',        'mean'),
    n_churners       = ('ever_churn',        'sum'),
    stress_medio     = ('stress_medio',      'mean'),
    impago_medio     = ('pct_meses_impago',  'mean'),
    ingreso_medio    = ('ingreso_estimado',  'mean'),
    antiguedad_media = ('antiguedad_meses',  'mean'),
    tipo_zona        = ('tipo_zona',         'first'),
    region           = ('region',            'first'),
).reset_index().sort_values('tasa_churn', ascending=False)

print(f'Zonas analizadas: {len(zona_stats)}')
print(f'Zona con mas churn: {zona_stats.iloc[0]["zona_id"]} ({zona_stats.iloc[0]["tasa_churn"]*100:.1f}%)')
print(f'Zona con menos churn: {zona_stats.iloc[-1]["zona_id"]} ({zona_stats.iloc[-1]["tasa_churn"]*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Barras horizontales: top 15 zonas
top_zonas = zona_stats.head(15)
colores   = sns.color_palette('RdYlGn_r', len(top_zonas))
bars = axes[0].barh(top_zonas['zona_id'], top_zonas['tasa_churn'] * 100, color=colores)
axes[0].set_title('Top 15 zonas por tasa de churn', fontweight='bold')
axes[0].set_xlabel('Tasa de churn (%)')
for bar, (_, row) in zip(bars, top_zonas.iterrows()):
    axes[0].text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{row['tasa_churn']*100:.1f}% ({row['tipo_zona']})",
        va='center', fontsize=8
    )
axes[0].set_xlim(0, zona_stats['tasa_churn'].max() * 130)

# Burbujas: stress vs tasa churn
scatter = axes[1].scatter(
    zona_stats['stress_medio'],
    zona_stats['tasa_churn'] * 100,
    s=zona_stats['n_clientes'] * 2,
    c=zona_stats['tasa_churn'],
    cmap='RdYlGn_r', alpha=0.7,
    edgecolors='white', linewidth=0.5
)
plt.colorbar(scatter, ax=axes[1], label='Tasa de churn')
for _, row in zona_stats[
    zona_stats['tasa_churn'] > zona_stats['tasa_churn'].quantile(0.75)
].iterrows():
    axes[1].annotate(
        row['zona_id'],
        (row['stress_medio'], row['tasa_churn'] * 100),
        fontsize=7, ha='center', va='bottom'
    )
axes[1].set_title(
    'Estres de red vs Tasa de churn por zona\n(tamano = n clientes)',
    fontweight='bold'
)
axes[1].set_xlabel('Estres de red medio')
axes[1].set_ylabel('Tasa de churn (%)')

plt.suptitle('Mapa de riesgo por zona geografica', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


Zonas analizadas: 31
Zona con mas churn: Z19 (39.4%)
Zona con menos churn: Z18 (12.4%)


In [ ]:
# Heatmap: tasa de churn por region x tipo de zona
pivot = df.groupby(['region', 'tipo_zona'])['ever_churn'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
    ax=ax, linewidths=0.5,
    cbar_kws={'label': 'Tasa de churn (%)'}
)
ax.set_title('Tasa de churn (%) - Region x Tipo de zona', fontweight='bold')
ax.set_xlabel('Tipo de zona')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

print('\nZonas de mayor riesgo combinado (rural + region con mas churn):')
display(
    zona_stats[zona_stats['tipo_zona'] == 'rural']
    [['zona_id', 'region', 'n_clientes', 'tasa_churn', 'stress_medio']]
    .sort_values('tasa_churn', ascending=False)
    .head(5)
    .round(3)
)


---
## 4. Simulador de riesgo por cliente

Esta seccion convierte el modelo en una herramienta practica: dado el perfil de un cliente concreto, calcula su probabilidad de churn.

Es especialmente util para la demostracion en vivo durante la presentacion.


In [ ]:
# Entrenamos el modelo simulador con variables estaticas del perfil
# (variables que cualquier comercial puede introducir manualmente)

FEATURES_SIM_NUM = [
    'edad', 'num_lineas', 'ingreso_estimado', 'antiguedad_meses',
    'descuento_activo', 'importe_medio', 'pct_meses_impago',
    'dias_retraso_medio', 'stress_medio', 'n_interacciones'
]
FEATURES_SIM_CAT = ['tipo_plan', 'tipo_zona', 'region', 'sexo', 'estado_civil']

df_sim = df.dropna(subset=FEATURES_SIM_NUM)
X_sim  = df_sim[FEATURES_SIM_NUM + FEATURES_SIM_CAT]
y_sim  = df_sim['ever_churn']

num_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler()),
])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='constant', fill_value='desconocido')),
    ('enc', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])
prep_sim = ColumnTransformer([
    ('num', num_pipe, FEATURES_SIM_NUM),
    ('cat', cat_pipe, FEATURES_SIM_CAT),
])

modelo_simulador = Pipeline([
    ('preprocessor', prep_sim),
    ('model', LogisticRegression(
        class_weight='balanced', C=0.1, penalty='l1',
        solver='saga', random_state=RANDOM_STATE, max_iter=1000
    ))
])
modelo_simulador.fit(X_sim, y_sim)

auc_sim = cross_val_score(
    modelo_simulador, X_sim, y_sim,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc'
).mean()
print(f'Modelo simulador entrenado - AUC CV: {auc_sim:.3f}')


Modelo simulador entrenado - AUC CV: 0.949


In [ ]:
def simular_cliente(perfil: dict) -> None:
    """
    Dado el perfil de un cliente, calcula su probabilidad de churn.

    Parametros del perfil:
        edad, num_lineas, ingreso_estimado, antiguedad_meses,
        descuento_activo, importe_medio, pct_meses_impago,
        dias_retraso_medio, stress_medio, n_interacciones,
        tipo_plan, tipo_zona, region, sexo, estado_civil
    """
    df_perfil = pd.DataFrame([perfil])
    proba = modelo_simulador.predict_proba(df_perfil)[0, 1]

    if proba >= 0.6:
        nivel  = 'ALTO'
        accion = 'Contacto prioritario - oferta de retencion inmediata'
    elif proba >= 0.3:
        nivel  = 'MEDIO'
        accion = 'Seguimiento proactivo - comunicacion en las proximas semanas'
    else:
        nivel  = 'BAJO'
        accion = 'No actuar - riesgo bajo de abandono'

    print('=' * 55)
    print('  RESULTADO DEL SIMULADOR DE RIESGO')
    print('=' * 55)
    print(f'  Probabilidad de churn:  {proba*100:.1f}%')
    print(f'  Nivel de riesgo:        {nivel}')
    print(f'  Accion recomendada:     {accion}')
    print('=' * 55)

    fig, ax = plt.subplots(figsize=(8, 1.5))
    color = '#E85C4C' if proba >= 0.6 else ('#f39c12' if proba >= 0.3 else '#27ae60')
    ax.barh(['Riesgo'], [100], color='#f0f0f0', height=0.4)
    ax.barh(['Riesgo'], [proba * 100], color=color, height=0.4)
    ax.axvline(30, color='#f39c12', linestyle='--', linewidth=1, label='Umbral medio (30%)')
    ax.axvline(60, color='#E85C4C', linestyle='--', linewidth=1, label='Umbral alto (60%)')
    ax.set_xlim(0, 100)
    ax.set_xlabel('Probabilidad de churn (%)')
    ax.set_title(f'Riesgo del cliente: {proba*100:.1f}%', fontweight='bold')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

print('Simulador definido. Usa simular_cliente({...}) para calcular el riesgo.')


Simulador definido. Usa simular_cliente({...}) para calcular el riesgo.


In [ ]:
# EJEMPLO 1 - Cliente de alto riesgo
print('EJEMPLO 1 - Cliente de alto riesgo:')
simular_cliente({
    'edad':               35,
    'num_lineas':         1,
    'ingreso_estimado':   1800,
    'antiguedad_meses':   8,
    'descuento_activo':   0,
    'importe_medio':      45,
    'pct_meses_impago':   0.40,
    'dias_retraso_medio': 12,
    'stress_medio':       0.65,
    'n_interacciones':    8,
    'tipo_plan':          'Prepago',
    'tipo_zona':          'rural',
    'region':             'Centro',
    'sexo':               'Hombre',
    'estado_civil':       'Soltero/a',
})


EJEMPLO 1 - Cliente de alto riesgo:
  RESULTADO DEL SIMULADOR DE RIESGO
  Probabilidad de churn:  100.0%
  Nivel de riesgo:        ALTO
  Accion recomendada:     Contacto prioritario - oferta de retencion inmediata


In [ ]:
# EJEMPLO 2 - Cliente de bajo riesgo
print('EJEMPLO 2 - Cliente de bajo riesgo:')
simular_cliente({
    'edad':               52,
    'num_lineas':         3,
    'ingreso_estimado':   4500,
    'antiguedad_meses':   48,
    'descuento_activo':   1,
    'importe_medio':      180,
    'pct_meses_impago':   0.02,
    'dias_retraso_medio': 1,
    'stress_medio':       0.25,
    'n_interacciones':    2,
    'tipo_plan':          'Premium',
    'tipo_zona':          'urbana_premium',
    'region':             'Este',
    'sexo':               'Mujer',
    'estado_civil':       'Casado/a',
})


EJEMPLO 2 - Cliente de bajo riesgo:
  RESULTADO DEL SIMULADOR DE RIESGO
  Probabilidad de churn:  70.1%
  Nivel de riesgo:        ALTO
  Accion recomendada:     Contacto prioritario - oferta de retencion inmediata


---
## 5. Modelo por segmento de plan

Hasta ahora hemos entrenado un modelo global para todos los clientes. Pero un cliente Prepago tiene patrones de churn muy distintos a uno Premium:
- **Prepago**: 26.1% de ever_churn — sin compromiso contractual, muy sensibles al precio
- **Contrato**: 21.2% de ever_churn — con algun vinculo pero pueden romperlo
- **Premium**: 16.1% de ever_churn — mayor valor, mas fidelizados

Hacemos el analisis en dos versiones:
- **5a - Modelo binario**: rapido, para ver si el AUC mejora por segmento. Tiene leakage (misma limitacion que el modelo binario global).
- **5b - Modelo temporal**: el correcto para produccion, con lags para evitar leakage.


### 5a - Modelo binario por segmento


In [ ]:
FEATURES_SEG_NUM = [
    'edad', 'num_lineas', 'ingreso_estimado', 'antiguedad_meses', 'descuento_activo',
    'importe_medio', 'pct_meses_impago', 'dias_retraso_medio',
    'stress_medio', 'n_interacciones'
]
FEATURES_SEG_CAT = ['tipo_zona', 'region', 'sexo', 'estado_civil']

prep_seg = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), FEATURES_SEG_NUM),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='desconocido')),
        ('enc', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), FEATURES_SEG_CAT),
])

def modelo_lr(prep):
    return Pipeline([
        ('prep',  prep),
        ('model', LogisticRegression(
            class_weight='balanced', C=0.1,
            penalty='l1', solver='saga',
            random_state=RANDOM_STATE, max_iter=1000
        ))
    ])

cv5 = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
res_binario = []

# Global
df_g  = df.dropna(subset=FEATURES_SEG_NUM)
auc_g = cross_val_score(
    modelo_lr(prep_seg),
    df_g[FEATURES_SEG_NUM + FEATURES_SEG_CAT],
    df_g['ever_churn'],
    cv=cv5, scoring='roc_auc'
).mean()
res_binario.append({'Segmento': 'Global', 'n': len(df_g),
                    'churn_%': df_g['ever_churn'].mean()*100, 'AUC_binario': auc_g})
print(f'Global    - AUC: {auc_g:.3f} | n={len(df_g):,}')

# Por segmento
for plan in ['Prepago', 'Contrato', 'Premium']:
    sub = df[df['tipo_plan'] == plan].dropna(subset=FEATURES_SEG_NUM)
    if sub['ever_churn'].sum() < 20:
        continue
    auc_s = cross_val_score(
        modelo_lr(prep_seg),
        sub[FEATURES_SEG_NUM + FEATURES_SEG_CAT],
        sub['ever_churn'],
        cv=cv5, scoring='roc_auc'
    ).mean()
    res_binario.append({'Segmento': plan, 'n': len(sub),
                        'churn_%': sub['ever_churn'].mean()*100, 'AUC_binario': auc_s})
    print(f'{plan:10s} - AUC: {auc_s:.3f} | n={len(sub):,} | churn={sub["ever_churn"].mean()*100:.1f}%')

df_bin = pd.DataFrame(res_binario)
print()
display(df_bin.round(3))


Global     - AUC: 0.947 | n=9,100
Prepago    - AUC: 0.946 | n=2,427 | churn=25.9%
Contrato   - AUC: 0.946 | n=2,053 | churn=20.8%
Premium    - AUC: 0.941 | n=4,620 | churn=16.1%


### 5b - Modelo temporal por segmento

Version correcta para produccion. Construimos el panel temporal con lags para cada segmento y comparamos con el AUC temporal global (**0.701**).

Datos por segmento en el panel temporal:
- **Prepago**: 80,100 filas | 0.87% tasa mensual
- **Contrato**: 69,815 filas | 0.69% tasa mensual
- **Premium**: 162,072 filas | 0.50% tasa mensual


In [ ]:
# Panel temporal completo (mismo que modelo_features)
factura['mes'] = factura['fecha'].dt.to_period('M').dt.to_timestamp()
fac_mes = factura.groupby(['cliente_id', 'mes']).agg(
    importe_mes       = ('importe_total',        'sum'),
    impago_mes        = ('impago_flag',           'max'),
    dias_retraso_mes  = ('dias_retraso_pago',     'max'),
    stress_mes        = ('stress_calidad_lag',    'mean'),
    variacion_consumo = ('variacion_consumo_pct', 'mean'),
).reset_index().sort_values(['cliente_id', 'mes'])

# Rolling 3 meses
for col in ['impago_mes', 'stress_mes', 'dias_retraso_mes']:
    fac_mes[col + '_roll3'] = (
        fac_mes.groupby('cliente_id')[col]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

# Racha de impagos
impago_lag = fac_mes.groupby('cliente_id')['impago_mes'].shift(1).fillna(0)
grupos     = (
    impago_lag != impago_lag.groupby(fac_mes['cliente_id']).shift(1)
).cumsum()
fac_mes['racha_impagos'] = impago_lag * fac_mes.groupby(grupos).cumcount().add(1)

# Lag 1
cols_lag = [
    'importe_mes', 'impago_mes', 'dias_retraso_mes', 'stress_mes',
    'variacion_consumo', 'impago_mes_roll3', 'stress_mes_roll3',
    'dias_retraso_mes_roll3', 'racha_impagos'
]
fac_lag = fac_mes[['cliente_id', 'mes'] + cols_lag].copy()
fac_lag['mes'] = fac_lag['mes'] + pd.DateOffset(months=1)
fac_lag = fac_lag.rename(columns={c: c + '_lag1' for c in cols_lag})

# Panel base
panel_t = churn.copy()
panel_t['mes'] = panel_t['fecha'].dt.to_period('M').dt.to_timestamp()
panel_t = panel_t.merge(fac_lag, on=['cliente_id', 'mes'], how='left')

cols_perfil = [
    'cliente_id', 'edad', 'sexo', 'estado_civil', 'num_lineas',
    'tipo_plan', 'tipo_zona', 'region', 'ingreso_estimado',
    'antiguedad_meses', 'descuento_activo'
]
panel_t = panel_t.merge(
    clientes[[c for c in cols_perfil if c in clientes.columns]],
    on='cliente_id', how='left'
)
panel_t['tipo_plan_num'] = panel_t['tipo_plan'].map(
    {'Basico': 1, 'Estandar': 2, 'Prepago': 1, 'Contrato': 2, 'Premium': 3}
)

FEAT_T_NUM = [
    'edad', 'num_lineas', 'ingreso_estimado', 'antiguedad_meses', 'descuento_activo',
    'importe_mes_lag1', 'impago_mes_lag1', 'dias_retraso_mes_lag1', 'stress_mes_lag1',
    'variacion_consumo_lag1', 'impago_mes_roll3_lag1', 'stress_mes_roll3_lag1',
    'dias_retraso_mes_roll3_lag1', 'racha_impagos_lag1',
]
FEAT_T_CAT = ['tipo_zona', 'region', 'sexo', 'estado_civil']
FEAT_T_ORD = ['tipo_plan_num']
all_feat_t  = FEAT_T_NUM + FEAT_T_CAT + FEAT_T_ORD

prep_t = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), FEAT_T_NUM),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='desconocido')),
        ('enc', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), FEAT_T_CAT),
    ('ord', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), FEAT_T_ORD),
])

res_temporal = []

# Global temporal (referencia)
panel_g  = panel_t.dropna(subset=['importe_mes_lag1', 'stress_mes_lag1'])
cli_u    = panel_g['cliente_id'].unique()
np.random.seed(RANDOM_STATE)
cli_test  = np.random.choice(cli_u, size=int(len(cli_u) * 0.2), replace=False)
cli_train = np.setdiff1d(cli_u, cli_test)
train_g   = panel_g[panel_g['cliente_id'].isin(cli_train)]
test_g    = panel_g[panel_g['cliente_id'].isin(cli_test)]

pipe_g = Pipeline([
    ('prep', prep_t),
    ('model', LogisticRegression(
        class_weight='balanced', C=0.1, penalty='l1',
        solver='saga', random_state=RANDOM_STATE, max_iter=1000
    ))
])
pipe_g.fit(train_g[all_feat_t], train_g['churn'])
auc_g_t = roc_auc_score(
    test_g['churn'],
    pipe_g.predict_proba(test_g[all_feat_t])[:, 1]
)
res_temporal.append({'Segmento': 'Global temporal', 'n_filas': len(panel_g),
                     'churn_%': panel_g['churn'].mean()*100, 'AUC_temporal': auc_g_t})
print(f'Global temporal - AUC test: {auc_g_t:.3f}')

# Por segmento temporal
for plan in ['Prepago', 'Contrato', 'Premium']:
    sub = panel_t[panel_t['tipo_plan'] == plan].dropna(
        subset=['importe_mes_lag1', 'stress_mes_lag1']
    )
    if sub['churn'].sum() < 50:
        print(f'{plan}: insuficientes churners, omitido')
        continue

    cli_s   = sub['cliente_id'].unique()
    np.random.seed(RANDOM_STATE)
    c_test  = np.random.choice(cli_s, size=int(len(cli_s) * 0.2), replace=False)
    c_train = np.setdiff1d(cli_s, c_test)

    tr = sub[sub['cliente_id'].isin(c_train)]
    te = sub[sub['cliente_id'].isin(c_test)]

    pipe_s = Pipeline([
        ('prep', prep_t),
        ('model', LogisticRegression(
            class_weight='balanced', C=0.1, penalty='l1',
            solver='saga', random_state=RANDOM_STATE, max_iter=1000
        ))
    ])
    pipe_s.fit(tr[all_feat_t], tr['churn'])
    auc_s = roc_auc_score(
        te['churn'],
        pipe_s.predict_proba(te[all_feat_t])[:, 1]
    )
    res_temporal.append({'Segmento': plan, 'n_filas': len(sub),
                         'churn_%': sub['churn'].mean()*100, 'AUC_temporal': auc_s})
    print(f'{plan:10s} - AUC test: {auc_s:.3f} | '
          f'filas={len(sub):,} | churn={sub["churn"].mean()*100:.2f}%')

df_temp = pd.DataFrame(res_temporal)
print()
display(df_temp.round(3))


Global temporal - AUC test: 0.718
Prepago     - AUC test: 0.733 | filas=80,100 | churn=0.87%
Contrato    - AUC test: 0.684 | filas=69,815 | churn=0.69%
Premium     - AUC test: 0.673 | filas=162,072 | churn=0.50%


In [ ]:
# Grafico comparativo: binario vs temporal por segmento
df_comp = df_bin.rename(columns={'AUC_binario': 'Binario'}).merge(
    df_temp.rename(columns={'AUC_temporal': 'Temporal'})
    [['Segmento', 'Temporal']]
    .assign(Segmento=lambda x: x['Segmento'].str.replace(' temporal', '')),
    on='Segmento', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x     = np.arange(len(df_comp))
width = 0.35
axes[0].bar(x - width/2, df_comp['Binario'],  width, label='Binario',  color='#a8c6e8', alpha=0.9)
axes[0].bar(x + width/2, df_comp['Temporal'], width, label='Temporal', color='#E85C4C', alpha=0.9)
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_comp['Segmento'], rotation=10)
axes[0].set_ylim(0.5, 0.85)
axes[0].axhline(0.701, color='gray', linestyle='--',
                label='Mejor modelo temporal anterior (0.701)')
axes[0].set_title('AUC: Binario vs Temporal por segmento', fontweight='bold')
axes[0].set_ylabel('AUC-ROC')
axes[0].legend(fontsize=8)
for i, (_, row) in enumerate(df_comp.iterrows()):
    if pd.notna(row.get('Binario')):
        axes[0].text(i - width/2, row['Binario'] + 0.003,
                     f'{row["Binario"]:.3f}', ha='center', fontsize=8)
    if pd.notna(row.get('Temporal')):
        axes[0].text(i + width/2, row['Temporal'] + 0.003,
                     f'{row["Temporal"]:.3f}', ha='center', fontsize=8)

tasas = {'Global': 20.0, 'Prepago': 26.1, 'Contrato': 21.2, 'Premium': 16.1}
colores_plan = ['#4C9BE8', '#f39c12', '#27ae60', '#95a5a6']
segs  = [s for s in df_comp['Segmento'] if s in tasas]
bars2 = axes[1].bar(segs, [tasas[s] for s in segs], color=colores_plan[:len(segs)])
axes[1].set_title('Tasa de churn por segmento (ever_churn)', fontweight='bold')
axes[1].set_ylabel('% clientes con churn')
for bar, seg in zip(bars2, segs):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{tasas[seg]:.1f}%',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

plt.suptitle('Modelo por segmento - Binario vs Temporal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 6. Conclusiones

### Perfil del churner tipico

El cliente en riesgo es de **plan Premium, zona suburbana, region Oeste**, con una antiguedad mediana de **24 meses**, impagos en el **14.8%** de sus meses y un estres de red medio de **0.458**. Las senales mas claras son:
- **Impago y retraso en el pago**: +48% y +59% respecto a no churners
- **Estres de red**: +18% — la calidad de la red es un factor diferenciador
- **Antiguedad e ingresos**: ambos menores en churners (-18% y -15%), los clientes mas nuevos y con menos recursos son los mas volatiles

### Mapa de riesgo por zona

- La zona **Z19** (Este, rural) lidera con un 39.4% de churn y estres de red 0.661
- Las zonas rurales de la region Este son consistentemente las de mayor riesgo (39.0% de media), mientras que las urbana_premium tienen tasas del 13.7-15.7%
- El scatter stress vs churn confirma la correlacion: a mayor estres de red, mayor tasa de abandono

### Simulador

- El AUC del modelo simulador en CV es **0.949** — alto porque usa agregados historicos (mismo leakage que el modelo binario, intencionado para la demo)
- Ejemplo 1 (Prepago, rural, 8 meses, 40% impagos): **100.0% de probabilidad** — ALTO
- Ejemplo 2 (Premium, urbana_premium, 48 meses, sin impagos): **70.1%** — sorprendentemente alto, probablemente porque el modelo binario captura sesgos de clase

### Modelo por segmento — 5a (Binario)

Los AUCs binarios son muy similares entre segmentos (0.941-0.947), lo que indica que el leakage afecta a todos los segmentos por igual y no hay informacion diferencial.

### Modelo por segmento — 5b (Temporal)

| Segmento | AUC Temporal | Tasa churn mensual |
|----------|-------------|--------------------|
| Global | 0.718 | 0.635% |
| **Prepago** | **0.733** | **0.87%** |
| Contrato | 0.684 | 0.69% |
| Premium | 0.673 | 0.50% |

**Conclusion clave**: el segmento **Prepago supera el AUC global (0.733 > 0.718)**, lo que justifica desplegar un modelo especializado para este segmento en produccion. Como era de esperar, Prepago tiene la tasa de churn mensual mas alta (0.87%) y patrones mas diferenciados. Contrato y Premium quedan por debajo del global, posiblemente porque sus churners son mas dificiles de anticipar con datos de comportamiento.

### Resumen de la evolucion del AUC en el proyecto

| Fase | Modelo | AUC | Nota |
|------|--------|-----|------|
| 1 | Binario LR | 0.991 | Leakage |
| 2 | Temporal LR | 0.685 | Sin leakage |
| 3 | LR GridSearch | 0.690 | Optimizacion |
| 4 | LR + Tendencia | 0.701 | Feature engineering |
| 5 | LR + V2 | 0.703 | Features multicausales |
| 6 | **Segmentado Prepago** | **0.733** | **Modelo especializado** |

---
*Prediccion de Churn - Empresa de Telecomunicaciones | Practicas Aplicadas 2026*
